## Step 3 — create building tessellation
**# of cells in notebook:** 1

**Purpose:** Create a building-level morphological tessellation within each selected block. Each tessellation cell is associated with one building and partitions the block into building-specific spatial units for use in later building-context calculations.

**Input:**

- the `heterogeneous_largePop_blocks` directory created in Step 2
- within each selected block folder, a same-named file geodatabase containing:
  - `block` — the individual selected block feature
  - `buildings` — buildings clipped to that block

**Output:**

Within each block folder:

- `building_level_tessellation.gpkg` — the building-level morphological tessellation, with `context_bldg_id` linking each cell to its source building

At the base block directory:

- `building_level_tessellation_summary.csv` — QA summary for all processed blocks

**Main logic:**

**Cell 1 — Create building-level tessellation**

1. Finds the block-specific workspaces created in Step 2 and reads each block boundary and buildings layer.
2. Preserves the FileGDB feature ID as `context_bldg_id`, providing a stable link between source buildings and tessellation cells.
3. Cleans polygon geometry and defensively clips the buildings to the block boundary.
4. Checks for building geometries likely to disappear during the momepy inward offset and for positive-area building overlaps that remain after shrinking.
5. Runs `momepy.Tessellation` using the block boundary as the tessellation limit.
6. Clips and normalizes the resulting cells, checks building-to-cell correspondence, geometry validity, multipart cells, block coverage, overlaps, and outside-block area.
7. Writes `building_level_tessellation.gpkg` for each block and an overall QA summary CSV.


In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import momepy

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------------------

base_folder = r"E:\World Bank deliverbale 1\_analysis\heterogeneous_largePop_blocks"

# Momepy morphological-tessellation parameters
shrink = 0.4
segment = 0.5

# Output written inside each block folder
out_tess_gpkg_name = "building_level_tessellation.gpkg"
out_tess_layer = "building_level_tessellation"

# Overall QA summary written to base_folder
summary_csv_name = "building_level_tessellation_summary.csv"

overwrite_outputs = True


# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------

def clean_polygonal(g):
    """Repair geometry and retain polygonal content only."""
    if g is None or g.is_empty:
        return None

    try:
        g = shapely.make_valid(g)
    except Exception:
        try:
            g = g.buffer(0)
        except Exception:
            return None

    if g is None or g.is_empty:
        return None

    if g.geom_type == "GeometryCollection":
        parts = [
            part
            for part in g.geoms
            if part.geom_type in ("Polygon", "MultiPolygon") and not part.is_empty
        ]
        if not parts:
            return None
        g = shapely.union_all(parts)

    # FileGDB polygon layers are often exposed as one-part MultiPolygons.
    # Normalize those to Polygon so they do not create false multipart warnings.
    if g.geom_type == "MultiPolygon" and len(g.geoms) == 1:
        g = g.geoms[0]

    if g.geom_type not in ("Polygon", "MultiPolygon"):
        return None

    if not g.is_valid:
        try:
            g = g.buffer(0)
        except Exception:
            return None

    if g is None or g.is_empty:
        return None

    return g


def union_all_compat(geoseries):
    """GeoPandas-version-compatible union."""
    if hasattr(geoseries, "union_all"):
        return geoseries.union_all()
    return geoseries.unary_union


def find_block_folders(base_folder):
    """Find _<number> folders containing a same-named FileGDB."""
    block_folders = []

    for folder in sorted(glob.glob(os.path.join(base_folder, "_*"))):
        if not os.path.isdir(folder):
            continue

        folder_name = os.path.basename(folder)
        gdb_path = os.path.join(folder, f"{folder_name}.gdb")

        if os.path.isdir(gdb_path):
            block_folders.append(folder)

    return block_folders


def read_gdb_layer(gdb_path, layer_name, preserve_fid=False):
    """Read a FileGDB layer with GeoPandas/pyogrio."""
    kwargs = {"engine": "pyogrio"}

    if preserve_fid:
        kwargs["fid_as_index"] = True

    return gpd.read_file(gdb_path, layer=layer_name, **kwargs)


def likely_unrepresented_after_shrink(buildings, shrink, segment):
    """
    Predict IDs that momepy cannot represent after the inward offset.

    Momepy buffers polygon footprints inward by `shrink`, explodes multipart
    results, and only creates Voronoi input points for boundaries longer than
    `segment`. Very small clipped building slivers can therefore disappear.
    """
    ids = []

    for row in buildings[["context_bldg_id", "geometry"]].itertuples(index=False):
        g = row.geometry

        if shrink != 0:
            g = g.buffer(-shrink, cap_style=2, join_style=2)

        if g is None or g.is_empty:
            ids.append(int(row.context_bldg_id))
            continue

        if g.geom_type == "MultiPolygon":
            parts = list(g.geoms)
        else:
            parts = [g]

        eligible = [
            part
            for part in parts
            if not part.is_empty and part.boundary.length > segment
        ]

        if not eligible:
            ids.append(int(row.context_bldg_id))

    return ids


def overlap_pairs_after_shrink(buildings, shrink, area_tolerance=1e-9):
    """Return positive-area footprint overlaps after the momepy inward offset."""
    ids = []
    geoms = []

    for row in buildings[["context_bldg_id", "geometry"]].itertuples(index=False):
        g = row.geometry

        if shrink != 0:
            g = g.buffer(-shrink, cap_style=2, join_style=2)

        if g is None or g.is_empty:
            continue

        ids.append(int(row.context_bldg_id))
        geoms.append(g)

    if len(geoms) < 2:
        return [], 0.0

    arr = np.asarray(geoms, dtype=object)
    tree = shapely.STRtree(arr)
    query = tree.query(arr, predicate="intersects")

    overlap_pairs = []
    overlap_area = 0.0

    for left, right in zip(query[0], query[1]):
        if left >= right:
            continue

        inter = arr[left].intersection(arr[right])
        area = float(inter.area)

        if area > area_tolerance:
            overlap_pairs.append((ids[left], ids[right], area))
            overlap_area += area

    return overlap_pairs, overlap_area


def normalize_tessellation(tess, block_union):
    """Normalize momepy output, clip to the exact block, and enforce one row per ID."""
    tess = tess.copy()

    if "context_bldg_id" not in tess.columns:
        if tess.index.name == "context_bldg_id":
            tess = tess.reset_index()
        else:
            tess = tess.reset_index()

    if "context_bldg_id" not in tess.columns:
        raise ValueError("Could not find context_bldg_id in tessellation output.")

    tess["geometry"] = tess.geometry.intersection(block_union)
    tess["geometry"] = tess.geometry.apply(clean_polygonal)
    tess = tess[tess.geometry.notna() & ~tess.geometry.is_empty].copy()

    if len(tess) == 0:
        raise ValueError("Tessellation became empty after final clipping.")

    if tess["context_bldg_id"].duplicated().any():
        tess = tess[["context_bldg_id", "geometry"]].dissolve(
            by="context_bldg_id",
            as_index=False,
        )

    tess["context_bldg_id"] = tess["context_bldg_id"].astype("int64")

    return tess[["context_bldg_id", "geometry"]].copy()


# ---------------------------------------------------------------------
# FIND BLOCK WORKSPACES
# ---------------------------------------------------------------------

block_folders = find_block_folders(base_folder)

print("=" * 88)
print("BUILDING-LEVEL MORPHOLOGICAL TESSELLATION")
print("=" * 88)
print(f"Base folder: {base_folder}")
print(f"Block folders found: {len(block_folders):,}")
print(f"Momepy shrink: {shrink}")
print(f"Momepy segment: {segment}")

if not block_folders:
    raise FileNotFoundError(
        f"No block folders with matching FileGDBs were found under:\n{base_folder}"
    )


# ---------------------------------------------------------------------
# PROCESS EACH BLOCK
# ---------------------------------------------------------------------

summary_rows = []

for block_folder in block_folders:
    block_name = os.path.basename(block_folder)
    gdb_path = os.path.join(block_folder, f"{block_name}.gdb")
    out_tess_gpkg = os.path.join(block_folder, out_tess_gpkg_name)

    print("\n" + "=" * 88)
    print(f"Processing {block_name}")
    print("=" * 88)
    print(f"GDB: {gdb_path}")

    summary = {
        "block_folder": block_name,
        "status": "failed",
        "error": "",
        "n_input_buildings": np.nan,
        "n_valid_buildings": np.nan,
        "n_tessellation_cells": np.nan,
        "n_missing_cells": np.nan,
        "missing_context_bldg_ids": "",
        "n_likely_unrepresented_after_shrink": np.nan,
        "likely_unrepresented_context_bldg_ids": "",
        "n_overlap_pairs_after_shrink": np.nan,
        "overlap_pairs_after_shrink": "",
        "overlap_area_after_shrink_m2": np.nan,
        "n_invalid_cells": np.nan,
        "n_true_multipart_cells": np.nan,
        "multipart_context_bldg_ids": "",
        "block_area_m2": np.nan,
        "tessellation_union_area_m2": np.nan,
        "missing_area_m2": np.nan,
        "missing_area_pct": np.nan,
        "overlap_area_m2": np.nan,
        "outside_block_area_m2": np.nan,
        "output_gpkg": out_tess_gpkg,
    }

    try:
        # -------------------------------------------------------------
        # Read block-specific inputs created in Step 2
        # -------------------------------------------------------------

        buildings = read_gdb_layer(
            gdb_path,
            "buildings",
            preserve_fid=True,
        )
        block = read_gdb_layer(gdb_path, "block")

        summary["n_input_buildings"] = len(buildings)

        if len(buildings) == 0:
            raise ValueError("Buildings layer is empty.")

        if len(block) == 0:
            raise ValueError("Block layer is empty.")

        if buildings.crs is None or block.crs is None:
            raise ValueError("Buildings and block layers must have a defined CRS.")

        if buildings.crs.is_geographic:
            raise ValueError(
                f"Buildings must use a projected CRS for tessellation; found {buildings.crs}."
            )

        if block.crs != buildings.crs:
            print("Reprojecting block boundary to the buildings CRS.")
            block = block.to_crs(buildings.crs)

        # Use the original FileGDB feature ID as the stable building/tessellation link.
        buildings = buildings.copy()
        buildings["context_bldg_id"] = buildings.index.astype("int64")

        # -------------------------------------------------------------
        # Clean geometries and defensively clip buildings to the block
        # -------------------------------------------------------------

        buildings["geometry"] = buildings.geometry.apply(clean_polygonal)
        buildings = buildings[
            buildings.geometry.notna() & ~buildings.geometry.is_empty
        ].copy()

        block["geometry"] = block.geometry.apply(clean_polygonal)
        block = block[block.geometry.notna() & ~block.geometry.is_empty].copy()

        if len(block) == 0:
            raise ValueError("No valid block geometry remains after cleaning.")

        block_union = clean_polygonal(union_all_compat(block.geometry))

        if block_union is None or block_union.is_empty:
            raise ValueError("Could not construct a valid block boundary union.")

        buildings["geometry"] = buildings.geometry.intersection(block_union)
        buildings["geometry"] = buildings.geometry.apply(clean_polygonal)
        buildings = buildings[
            buildings.geometry.notna() & ~buildings.geometry.is_empty
        ].copy()

        if len(buildings) == 0:
            raise ValueError("No valid buildings remain after clipping to the block.")

        summary["n_valid_buildings"] = len(buildings)
        summary["block_area_m2"] = float(block_union.area)

        print(f"Input buildings: {int(summary['n_input_buildings']):,}")
        print(f"Valid buildings: {len(buildings):,}")
        print(f"Block area: {block_union.area:,.3f} m²")

        # -------------------------------------------------------------
        # Pre-tessellation QA
        # -------------------------------------------------------------

        likely_missing = likely_unrepresented_after_shrink(
            buildings,
            shrink=shrink,
            segment=segment,
        )

        overlap_pairs, overlap_after_shrink_area = overlap_pairs_after_shrink(
            buildings,
            shrink=shrink,
        )

        summary["n_likely_unrepresented_after_shrink"] = len(likely_missing)
        summary["likely_unrepresented_context_bldg_ids"] = "|".join(
            map(str, likely_missing)
        )
        summary["n_overlap_pairs_after_shrink"] = len(overlap_pairs)
        summary["overlap_pairs_after_shrink"] = "|".join(
            f"{a}-{b}" for a, b, _ in overlap_pairs
        )
        summary["overlap_area_after_shrink_m2"] = overlap_after_shrink_area

        if likely_missing:
            print(
                "WARNING: buildings likely to be unrepresented after momepy shrink: "
                + ", ".join(map(str, likely_missing))
            )

        if overlap_pairs:
            print(
                "WARNING: positive-area building overlaps remain after shrink: "
                + ", ".join(f"{a}-{b}" for a, b, _ in overlap_pairs)
            )

        # -------------------------------------------------------------
        # Run momepy morphological tessellation
        # -------------------------------------------------------------

        print("Running momepy.Tessellation...")

        tess_obj = momepy.Tessellation(
            buildings[["context_bldg_id", "geometry"]],
            unique_id="context_bldg_id",
            limit=block_union,
            shrink=shrink,
            segment=segment,
            verbose=False,
        )

        tess = normalize_tessellation(
            tess_obj.tessellation,
            block_union=block_union,
        )

        tess["block_folder"] = block_name
        tess = tess[["context_bldg_id", "block_folder", "geometry"]].copy()

        # -------------------------------------------------------------
        # Post-tessellation QA
        # -------------------------------------------------------------

        building_ids = set(buildings["context_bldg_id"].astype(int))
        tess_ids = set(tess["context_bldg_id"].astype(int))
        missing_ids = sorted(building_ids - tess_ids)

        invalid_cells = int((~tess.geometry.is_valid).sum())

        multipart_ids = []
        for row in tess[["context_bldg_id", "geometry"]].itertuples(index=False):
            if row.geometry.geom_type == "MultiPolygon" and len(row.geometry.geoms) > 1:
                multipart_ids.append(int(row.context_bldg_id))

        tess_union = union_all_compat(tess.geometry)
        union_area = float(tess_union.area)
        block_area = float(block_union.area)
        missing_area = block_area - union_area
        missing_pct = 100.0 * missing_area / block_area if block_area > 0 else np.nan
        output_overlap_area = max(
            0.0,
            float(tess.geometry.area.sum() - union_area),
        )
        outside_area = float(tess_union.difference(block_union).area)

        summary["n_tessellation_cells"] = len(tess)
        summary["n_missing_cells"] = len(missing_ids)
        summary["missing_context_bldg_ids"] = "|".join(map(str, missing_ids))
        summary["n_invalid_cells"] = invalid_cells
        summary["n_true_multipart_cells"] = len(multipart_ids)
        summary["multipart_context_bldg_ids"] = "|".join(map(str, multipart_ids))
        summary["tessellation_union_area_m2"] = union_area
        summary["missing_area_m2"] = missing_area
        summary["missing_area_pct"] = missing_pct
        summary["overlap_area_m2"] = output_overlap_area
        summary["outside_block_area_m2"] = outside_area

        # Numerical area tolerance for floating-point geometry operations.
        area_tolerance = max(1e-6, block_area * 1e-9)

        qa_flag = (
            len(missing_ids) > 0
            or len(overlap_pairs) > 0
            or invalid_cells > 0
            or len(multipart_ids) > 0
            or abs(missing_area) > area_tolerance
            or output_overlap_area > area_tolerance
            or outside_area > area_tolerance
        )

        summary["status"] = "success_with_qa_flag" if qa_flag else "success"

        print(f"Tessellation cells: {len(tess):,}")
        print(f"Missing building cells: {len(missing_ids):,}")
        print(f"Invalid cells: {invalid_cells:,}")
        print(f"True multipart cells: {len(multipart_ids):,}")
        print(f"Tessellation union area: {union_area:,.6f} m²")
        print(f"Block - tessellation area: {missing_area:.9f} m²")
        print(f"Output overlap area: {output_overlap_area:.9f} m²")
        print(f"Outside-block area: {outside_area:.9f} m²")

        # -------------------------------------------------------------
        # Write tessellation only
        # -------------------------------------------------------------

        if os.path.exists(out_tess_gpkg):
            if overwrite_outputs:
                os.remove(out_tess_gpkg)
            else:
                raise FileExistsError(
                    f"Output exists and overwrite_outputs=False: {out_tess_gpkg}"
                )

        tess.to_file(
            out_tess_gpkg,
            layer=out_tess_layer,
            driver="GPKG",
        )

        print(f"Wrote: {out_tess_gpkg}")
        print(f"Status: {summary['status']}")

    except Exception as e:
        summary["error"] = f"{type(e).__name__}: {e}"
        print(f"FAILED: {summary['error']}")

    summary_rows.append(summary)


# ---------------------------------------------------------------------
# WRITE OVERALL QA SUMMARY
# ---------------------------------------------------------------------

summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(base_folder, summary_csv_name)
summary_df.to_csv(summary_csv, index=False)

print("\n" + "=" * 88)
print("FINAL SUMMARY")
print("=" * 88)
print(f"Blocks processed: {len(summary_df):,}")
print(f"Success: {(summary_df['status'] == 'success').sum():,}")
print(
    "Success with QA flag: "
    f"{(summary_df['status'] == 'success_with_qa_flag').sum():,}"
)
print(f"Failed: {(summary_df['status'] == 'failed').sum():,}")
print(f"Input buildings: {summary_df['n_input_buildings'].fillna(0).sum():,.0f}")
print(
    "Tessellation cells: "
    f"{summary_df['n_tessellation_cells'].fillna(0).sum():,.0f}"
)
print(
    "Missing building cells: "
    f"{summary_df['n_missing_cells'].fillna(0).sum():,.0f}"
)
print(f"Summary CSV: {summary_csv}")

flagged = summary_df[summary_df["status"] != "success"]

if len(flagged):
    print("\nBlocks requiring QA review:")
    print(
        flagged[
            [
                "block_folder",
                "status",
                "n_input_buildings",
                "n_tessellation_cells",
                "n_missing_cells",
                "n_overlap_pairs_after_shrink",
                "n_invalid_cells",
                "n_true_multipart_cells",
                "error",
            ]
        ].to_string(index=False)
    )
